# KOSPI Direction Prediction — Logistic Regression (Extended Answer Key)

Using 29 macro-financial indicators for this month, classify whether **the KOSPI will rise (1) or fall (0) next month**. English version of *코스피지수_로짓해답_wget.ipynb*, cleaned up and with a fuller evaluation.

| Part | What we do |
|---|---|
| **A** | The original lab: build the target → L1 logistic regression → accuracy |
| **B** | Added: confusion matrix, precision, recall, F1, ROC AUC, baselines, scaling, repeated random splits, thresholds |
| **C** | Lecture examples: the 8-mouse ROC (AUC 0.906) and the three iris species |

**Changes from the original**
1. Uses the same `KOSPI_Index_EN.csv` as Day 1 (UTF-8, English column names, `Date` column).
2. The target-building cell **gives the same result no matter how many times you run it**. In the original, re-running `kospi = kospi[:-1]` keeps dropping rows. The saved output (157 rows, accuracy 0.525) came from a state missing the first two rows; a clean run gives 159 rows and accuracy 0.55.
3. Accuracy is **compared with baselines** and re-checked over **100 random splits**.

## 0. Setup

In [1]:
import os
FILE = 'KOSPI_Index_EN.csv'
if not os.path.exists(FILE):
    try:
        from google.colab import files
        print('Please upload', FILE)
        files.upload()
    except ImportError:
        raise FileNotFoundError(f'{FILE} not found in {os.getcwd()}')

In [2]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings('ignore')
print('scikit-learn', sklearn.__version__)

def l1_logit(C):
    """L1 logistic regression. From scikit-learn 1.8 the penalty argument is replaced by l1_ratio."""
    major, minor = map(int, sklearn.__version__.split('.')[:2])
    if (major, minor) >= (1, 8):
        return LogisticRegression(solver='liblinear', l1_ratio=1, C=C, random_state=0)
    return LogisticRegression(solver='liblinear', penalty='l1', C=C, random_state=0)

scikit-learn 1.8.0


# Part A — The original lab

## 1. Load the data

In [3]:
kospi = pd.read_csv(FILE)
print(kospi.shape)
kospi.head()

(160, 31)


,Date,FKI BSI,Housing Price Index,Construction BSI (Outlook),Unemployment Rate (SA %),Dishonored Bill Rate,Daily Exports YoY,Outbound Travelers YoY,VKOSPI,MSB 364D Yield,...,Goods Exports,Domestic Demand,GDP Deflator,Gross Saving Rate,Gross Investment Rate,CRB Commodity Futures,CRB Energy,CRB Industrials,CRB Precious Metals,KOSPI
0,2003-05,96.4,70.13,83.3,3.7,0.09,14.50,-37.03,25.50,4.06,...,1.6,-0.4,3.1,32.8,32.3,243.7,247.7,217.5,322.4,633.4
1,2003-06,90.3,70.83,86.7,3.7,0.08,10.56,-9.67,20.83,4.24,...,7.8,-0.4,4.0,33.3,32.3,247.6,255.7,247.3,326.4,669.9
2,2003-07,91.4,70.57,86.7,3.8,0.06,14.83,0.93,26.98,4.60,...,7.8,0.3,4.0,33.3,31.8,249.9,268.7,232.3,340.1,713.5
3,2003-08,109.6,69.99,92.4,3.9,0.06,24.13,2.85,28.33,4.66,...,7.8,0.3,4.0,33.3,31.8,255.3,283.3,256.6,364.1,759.5
4,2003-09,110.3,69.70,64.1,3.8,0.08,25.49,14.34,25.35,4.51,...,10.9,0.3,3.3,34.7,31.8,262.6,292.7,262.0,368.3,697.5


## 2. Target: 1 if the KOSPI rises next month

Each row pairs month-t indicators with the direction from t to t+1. The features never use another month's values, so every row is an independent observation and a shuffled split is appropriate. The cell builds a fresh DataFrame, so re-running it is safe.

In [4]:
def classify(current, future):
    if future > current:
        return 1
    else:
        return 0

df = kospi.copy()                                  # keep the original untouched
df['next'] = df['KOSPI'].shift(-1)
df = df.iloc[:-1].copy()                           # the last month has no next-month value
df['target'] = list(map(classify, df['KOSPI'], df['next']))
df[['Date', 'KOSPI', 'next', 'target']].head()

,Date,KOSPI,next,target
0,2003-05,633.4,669.9,1
1,2003-06,669.9,713.5,1
2,2003-07,713.5,759.5,1
3,2003-08,759.5,697.5,0
4,2003-09,697.5,782.4,1


In [5]:
X = df.drop(columns=['Date', 'KOSPI', 'next', 'target'])
y = df['target']
print(X.shape)
print(y.value_counts().rename({1: 'up', 0: 'down/flat'}))
print('share of up months: %.3f' % y.mean())

(159, 29)
target
up           91
down/flat    68
Name: count, dtype: int64
share of up months: 0.572


## 3. Split and fit (original settings)

In [6]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)   # default 75/25, shuffled
lr = l1_logit(C=10)
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)

In [7]:
from sklearn.metrics import accuracy_score
print('accuracy:', accuracy_score(y_test, y_pred))

accuracy: 0.55


# Part B — A fuller evaluation

## 4. Confusion matrix and metrics

In [8]:
from sklearn.metrics import (confusion_matrix, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report)
print('confusion matrix (rows = actual [down, up], columns = predicted [down, up])')
print(confusion_matrix(y_test, y_pred))
print('precision: %.3f' % precision_score(y_test, y_pred))
print('recall   : %.3f' % recall_score(y_test, y_pred))
print('F1       : %.3f' % f1_score(y_test, y_pred))
proba = lr.predict_proba(X_test)[:, 1]            # AUC uses probabilities
print('ROC AUC  : %.3f' % roc_auc_score(y_test, proba))

confusion matrix (rows = actual [down, up], columns = predicted [down, up])
[[ 9 11]
 [ 7 13]]
precision: 0.542
recall   : 0.650
F1       : 0.591
ROC AUC  : 0.555


In [9]:
print(classification_report(y_test, y_pred, target_names=['down', 'up']))

              precision    recall  f1-score   support

        down       0.56      0.45      0.50        20
          up       0.54      0.65      0.59        20

    accuracy                           0.55        40
   macro avg       0.55      0.55      0.55        40
weighted avg       0.55      0.55      0.55        40



## 5. Pitfall 1: compare with a baseline

Saying 'always up' is right as often as the test set's share of up months.

In [10]:
from sklearn.dummy import DummyClassifier
dummy = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
print('always majority class (up) accuracy:', accuracy_score(y_test, dummy.predict(X_test)))
print('share of up months in the test set :', y_test.mean())

always majority class (up) accuracy: 0.5
share of up months in the test set : 0.5


## 6. Pitfall 2: L1 regularization without standardizing

The penalty acts on coefficient size, so large-unit features (such as outbound travelers) get small coefficients and escape the penalty. Put the scaler in a pipeline.

In [11]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
pipe = make_pipeline(StandardScaler(), l1_logit(C=10)).fit(X_train, y_train)
print('accuracy after scaling: %.3f' % accuracy_score(y_test, pipe.predict(X_test)))
print('AUC after scaling     : %.3f' % roc_auc_score(y_test, pipe.predict_proba(X_test)[:, 1]))

accuracy after scaling: 0.550
AUC after scaling     : 0.565


## 7. Pitfall 3: trusting a single random split

The test set has only 40 months, so one split is one draw. Repeat the random 75/25 split 100 times (seeds 0–99) and compare with 'always up' on the same splits.

In [12]:
rows = []
for seed in range(100):
    Xa, Xb, ya, yb = train_test_split(X, y, random_state=seed)
    m = make_pipeline(StandardScaler(), l1_logit(C=10)).fit(Xa, ya)
    rows.append({'accuracy': accuracy_score(yb, m.predict(Xb)),
                 'AUC': roc_auc_score(yb, m.predict_proba(Xb)[:, 1]),
                 'always up': yb.mean()})
rep = pd.DataFrame(rows)
print(rep.describe().loc[['mean', 'std', 'min', 'max']].round(3))
print('splits where the model beats always-up:', (rep['accuracy'] > rep['always up']).sum(), 'of 100')
print('splits with AUC above 0.5            :', (rep['AUC'] > 0.5).sum(), 'of 100')

      accuracy    AUC  always up
mean     0.545  0.555      0.570
std      0.076  0.080      0.066
min      0.375  0.387      0.450
max      0.725  0.752      0.725
splits where the model beats always-up: 40 of 100
splits with AUC above 0.5            : 73 of 100


### Choosing C with repeated stratified cross-validation

In [13]:
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=20, random_state=0)
for C in [0.01, 0.1, 1, 10]:
    m = make_pipeline(StandardScaler(), l1_logit(C))
    auc = cross_val_score(m, X, y, cv=cv, scoring='roc_auc')
    print(f'C={C:<5} AUC mean {auc.mean():.3f}  range {auc.min():.2f}–{auc.max():.2f}')

C=0.01  AUC mean 0.500  range 0.50–0.50


C=0.1   AUC mean 0.464  range 0.30–0.71


C=1     AUC mean 0.580  range 0.34–0.77


C=10    AUC mean 0.562  range 0.31–0.75


The 0.55 from section 3 is one draw. Over 100 random splits accuracy averages 0.545, below always-up at 0.570, and the model beats always-up in only 40 of them; even the best C gives a mean AUC of about 0.58. 29 macro indicators cannot reliably call next month's KOSPI direction. For 55% accuracy to mean anything it must (1) beat the baseline, (2) hold across other random splits, and (3) survive trading costs.

## 8. Changing the threshold

`predict` uses a 0.5 threshold. Working from the probabilities, a different threshold gives a different confusion matrix. We reuse the scaled model from section 6 on the same test set.

In [14]:
p = pipe.predict_proba(X_test)[:, 1]
for t in [0.3, 0.5, 0.7]:
    pred = (p >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    print(f'threshold {t}: TP={tp:2d} FP={fp:2d} FN={fn:2d} TN={tn:2d}  recall={tp/(tp+fn):.2f}  precision={tp/max(tp+fp,1):.2f}')

threshold 0.3: TP=18 FP=13 FN= 2 TN= 7  recall=0.90  precision=0.58
threshold 0.5: TP=13 FP=11 FN= 7 TN= 9  recall=0.65  precision=0.54
threshold 0.7: TP=10 FP=10 FN=10 TN=10  recall=0.50  precision=0.50


# Part C — Lecture examples

## 9. The 8-mouse ROC (AUC 0.9 in the lecture)

In [15]:
from sklearn.metrics import roc_curve
y_mice = np.array([0, 0, 0, 0, 1, 1, 1, 1])                       # 0 = not obese, 1 = obese
score  = np.array([0.03, 0.06, 0.30, 0.55, 0.30, 0.85, 0.93, 0.97])   # predicted probability of obesity
fpr, tpr, thr = roc_curve(y_mice, score)
print(pd.DataFrame({'threshold': thr, 'FPR': fpr, 'TPR': tpr}).round(3))
print('AUC =', roc_auc_score(y_mice, score))

   threshold   FPR   TPR
0        inf  0.00  0.00
1       0.97  0.00  0.25
2       0.85  0.00  0.75
3       0.55  0.25  0.75
4       0.30  0.50  1.00
5       0.03  1.00  1.00
AUC = 0.90625


## 10. Three iris species (Lab 1 in the lecture)

In [16]:
from sklearn.datasets import load_iris
iris = load_iris(as_frame=True)
Xi_train, Xi_test, yi_train, yi_test = train_test_split(iris.data, iris.target, test_size=0.2, random_state=1)
clf = LogisticRegression(max_iter=1000).fit(Xi_train, yi_train)     # multiclass: softmax (multinomial)
yi_pred = clf.predict(Xi_test)
print('accuracy: {:.2f}%'.format(accuracy_score(yi_test, yi_pred) * 100))
print('coefficient shape:', clf.coef_.shape)
print(confusion_matrix(yi_test, yi_pred))
print('class probabilities for the first sample:', clf.predict_proba(Xi_test.iloc[:1]).round(3), '(sum = 1)')

accuracy: 96.67%
coefficient shape: (3, 4)
[[11  0  0]
 [ 0 12  1]
 [ 0  0  6]]
class probabilities for the first sample: [[0.985 0.015 0.   ]] (sum = 1)


## 11. Exercises
1. Change the target to 'next month's return ≤ −5%' (a sharp fall). What is the positive share? Which metric should replace accuracy?
2. Choose C with `LogisticRegressionCV(cv=StratifiedKFold(5), Cs=10)` and compare the result with the baselines.
3. In the 8-mouse example, which threshold maximizes the Youden index J = TPR − FPR?